In [1]:
%load_ext sql
%sql sqlite:///data/cookie_cats.db

Connecting to 'sqlite:///data/cookie_cats.db'

In [2]:
import sqlite3, pandas as pd
df = pd.read_csv("data/cookie_cats.csv")
con = sqlite3.connect("data/cookie_cats.db")
df.to_sql("players", con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql_query(sql, con)

q("SELECT * FROM players LIMIT 5")

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,0,0
1,337,gate_30,38,1,0
2,377,gate_40,165,1,0
3,483,gate_40,1,0,0
4,488,gate_40,179,1,1


In [3]:
q("SELECT COUNT(*) AS n_players FROM players")


,n_players
0,90189


In [14]:
q("SELECT COUNT(DISTINCT userid) AS unique_players, COUNT(*) AS total_players FROM players")

,unique_players,total_players
0,90189,90189


There are no duplicate players in the data set. Each row indicated a unique player. 

In [5]:
q('SELECT version, COUNT(*) AS n_players FROM players GROUP BY version')

,version,n_players
0,gate_30,44700
1,gate_40,45489


We can see the version usage of this moble game is roughly 50/50 with slightly more people favoring the newer version.

In [11]:
%%sql
SELECT MAX(sum_gamerounds) AS max_rounds, MIN(sum_gamerounds) AS min_rounds, AVG(sum_gamerounds) AS avg_rounds
FROM players

Running query in 'sqlite:///data/cookie_cats.db'

max_rounds,min_rounds,avg_rounds
49854,0,51.8724567297564


There is a difinitive skew here. The average rounds is much smaller than the maximum and there are people in the data set that haven't even played any rounds of the game. 

In [13]:
%%sql 
SELECT COUNT(*) AS zero_rounds
FROM players
WHERE sum_gamerounds == 0

Running query in 'sqlite:///data/cookie_cats.db'

zero_rounds
3994


This tells us that the dataset consists of users that haven't even played the game. They should be omited from any trends or analizations. 

In [19]:
%%sql 
WITH daily AS (
    SELECT userid, version, sum_gamerounds, ROUND(sum_gamerounds/ 14.0, 2) AS rounds_per_day 
    FROM players 
)
SELECT *, ROUNd(rounds_per_day * 3.0 , 2) AS minutes_spent_per_day
FROM daily
ORDER BY sum_gamerounds DESC 
LIMIT 20


Running query in 'sqlite:///data/cookie_cats.db'

userid,version,sum_gamerounds,rounds_per_day,minutes_spent_per_day
6390605,gate_30,49854,3561.0,10683.0
871500,gate_30,2961,211.5,634.5
3271615,gate_40,2640,188.57,565.71
4832608,gate_30,2438,174.14,522.42
5346171,gate_40,2294,163.86,491.58
5133952,gate_30,2251,160.79,482.37
9640085,gate_30,2156,154.0,462.0
4090246,gate_40,2124,151.71,455.13
9791599,gate_40,2063,147.36,442.08
725080,gate_40,2015,143.93,431.79


We can see here than the maximum of the data set is an extreme outlyer. Since the data is the usage of the game over the course of 2 weeks, it means that the user would have been using the app for roughly 10683 mintues per day. This is impossible as there are only 1440 minutes in a day. This is an error in the data set and needs to be omiited as well. 

In [37]:
%%sql 
DROP VIEW players_clean

Running query in 'sqlite:///data/cookie_cats.db'

++
||
++
++

In [39]:
%%sql 
CREATE VIEW players_clean AS 
SELECT *
FROM players 
WHERE sum_gamerounds != 0 AND sum_gamerounds < (1440/3) * 14




Running query in 'sqlite:///data/cookie_cats.db'

++
||
++
++

In [44]:
%%sql
SELECT
    (SELECT COUNT(*) FROM players)       AS raw_rows,
    (SELECT COUNT(*) FROM players_clean) AS clean_rows

Running query in 'sqlite:///data/cookie_cats.db'

raw_rows,clean_rows
90189,86194


The cleaned rows include only players that have played at least one round and the players that could have feasably played all of their rounds in 2 weeks.

In [47]:

%%sql
SELECT MAX(sum_gamerounds) AS max_rounds, MIN(sum_gamerounds) AS min_rounds, ROUND(AVG(sum_gamerounds), 2) AS avg_rounds
FROM players_clean

Running query in 'sqlite:///data/cookie_cats.db'

max_rounds,min_rounds,avg_rounds
2961,1,53.7


We can see here that the maximum is much more feasable, but suprsinly the average is still the same. This tells us there must be many players that have played very few rounds compared to the high round players. 

In [49]:

%%sql
SELECT sum_gamerounds 
FROM players_clean
ORDER BY sum_gamerounds ASC
LIMIT 1 
OFFSET 86194/2

Running query in 'sqlite:///data/cookie_cats.db'

sum_gamerounds
18


We can see here that the median of the data set is actually much lower than the mean. With at median of 18 it is evident that majority of the players in the data set are playing a small amount of rounds compared to the top players. 

In [59]:
%%sql
SELECT version,
       COUNT(*) AS n_players,
       ROUND(AVG(retention_1) * 100, 2) AS retention_1_percent,
       ROUND(AVG(retention_7) * 100, 2) AS retention_7_percent
FROM players_clean
GROUP BY version

Running query in 'sqlite:///data/cookie_cats.db'

version,n_players,retention_1_percent,retention_7_percent
gate_30,42762,46.75,19.84
gate_40,43432,46.22,19.03


The verson gate_30 has a slightly better retention rate on day 1 and day 7.

In [67]:
%%sql 
SELECT version, SUM(retention_7) AS retention_7_count, SUM(retention_1) AS retention_1_count
FROM players_clean
GROUP BY version


Running query in 'sqlite:///data/cookie_cats.db'

version,retention_7_count,retention_1_count
gate_30,8485,19993
gate_40,8266,20073


In [69]:
%pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 32.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [88]:
from scipy.stats import chi2_contingency

day7 = [
    [8485, 42762 - 8485],
    [8266, 43432 - 8266],
]

chi2, p, dof, expected = chi2_contingency(day7)
print(f"Day 7: chi2 = {chi2:.2f}, p = {p:.4f}")

Day 7: chi2 = 8.98, p = 0.0027


In [89]:


day1 = [
    [19993, 42762 - 19993],
    [20073, 43432 - 20073],
]

chi2, p, dof, expected = chi2_contingency(day1)
print(f"Day 1: chi2 = {chi2:.2f}, p = {p:.4f}")

Day 1: chi2 = 2.48, p = 0.1155


Day-7 retention was significantly higher for gate_30 than for gate_40 (19.84% vs. 19.03%, p = 0.003). That is a 0.81 percentage-point gap, or roughly a 4% relative drop in players returning after a week when the gate is moved to level 40. On day 1, there was no significant difference (46.75% vs. 46.22%, p = 0.12), using a significance threshold of p < 0.05.

This pattern makes sense. On day 1, most players likely had not played long enough to reach the gate, but by day 7 many had. Based on this data, I would recommend keeping the gate at level 30, since it produced a statistically significant higher 7-day retention rate.

One limitation: many players never reached level 30, so for them the two versions were identical. This likely dilutes the measured effect, meaning the true impact on players who reached the gate may be larger.

In [80]:
%%sql
SELECT version,
    CASE
            WHEN sum_gamerounds <= 10  THEN '1-10'
            WHEN sum_gamerounds <= 29  THEN '11-29'
            WHEN sum_gamerounds <= 39  THEN '30-39'
            WHEN sum_gamerounds <= 99  THEN '40-99'
            ELSE '100+'
    END AS rounds_bucket,
    COUNT(*) AS n_players
FROM players_clean 
GROUP BY version, rounds_bucket
ORDER BY version, MIN(sum_gamerounds)


Running query in 'sqlite:///data/cookie_cats.db'

version,rounds_bucket,n_players
gate_30,1-10,15736
gate_30,11-29,10371
gate_30,30-39,3090
gate_30,40-99,7398
gate_30,100+,6167
gate_40,1-10,16259
gate_40,11-29,10560
gate_40,30-39,2786
gate_40,40-99,7479
gate_40,100+,6348


In [84]:
%%sql
SELECT version,
       CASE
           WHEN sum_gamerounds <= 10 THEN '1-10'
           WHEN sum_gamerounds <= 29 THEN '11-29'
           WHEN sum_gamerounds <= 39 THEN '30-39'
           WHEN sum_gamerounds <= 99 THEN '40-99'
           ELSE '100+'
       END AS rounds_bucket,
       COUNT(*) AS n_players,
       ROUND(AVG(retention_7) * 100, 2) AS retention_7_percent
FROM players_clean
GROUP BY version, rounds_bucket
ORDER BY version, MIN(sum_gamerounds)



Running query in 'sqlite:///data/cookie_cats.db'

version,rounds_bucket,n_players,retention_7_percent
gate_30,1-10,15736,1.93
gate_30,11-29,10371,8.43
gate_30,30-39,3090,15.73
gate_30,40-99,7398,32.9
gate_30,100+,6167,71.14
gate_40,1-10,16259,1.94
gate_40,11-29,10560,7.64
gate_40,30-39,2786,15.69
gate_40,40-99,7479,29.39
gate_40,100+,6348,71.03


In [86]:
import json

retention = q("""SELECT version,
       COUNT(*) AS n_players,
       ROUND(AVG(retention_1) * 100, 2) AS retention_1_percent,
       ROUND(AVG(retention_7) * 100, 2) AS retention_7_percent
FROM players_clean
GROUP BY version""")
buckets = q("""SELECT version,
    CASE
            WHEN sum_gamerounds <= 10  THEN '1-10'
            WHEN sum_gamerounds <= 29  THEN '11-29'
            WHEN sum_gamerounds <= 39  THEN '30-39'
            WHEN sum_gamerounds <= 99  THEN '40-99'
            ELSE '100+'
    END AS rounds_bucket,
    COUNT(*) AS n_players
FROM players_clean 
GROUP BY version, rounds_bucket
ORDER BY version, MIN(sum_gamerounds)""")
retention_by_bucket = q("""SELECT version,
       CASE
           WHEN sum_gamerounds <= 10 THEN '1-10'
           WHEN sum_gamerounds <= 29 THEN '11-29'
           WHEN sum_gamerounds <= 39 THEN '30-39'
           WHEN sum_gamerounds <= 99 THEN '40-99'
           ELSE '100+'
       END AS rounds_bucket,
       COUNT(*) AS n_players,
       ROUND(AVG(retention_7) * 100, 2) AS retention_7_percent
FROM players_clean
GROUP BY version, rounds_bucket
ORDER BY version, MIN(sum_gamerounds)""")

data = {
    "retention": retention.to_dict(orient="records"),
    "buckets": buckets.to_dict(orient="records"),
    "retention_by_bucket": retention_by_bucket.to_dict(orient="records"),
}

with open("dashboard/data.json", "w") as f:
    json.dump(data, f, indent=2)